# Final Assignment: Analyzing Historical Stock/Revenue Data and Building a Dashboard

## Overview
As a Data Scientist / Analyst working for a startup investment firm, your task is to extract financial data (historical stock prices and quarterly revenue) for key stocks (**Tesla, GameStop, Amazon, AMD**) using Python libraries (`yfinance`, `BeautifulSoup`, `requests`, `pandas`) and web scraping techniques. You will then visualize this data in interactive dashboards using **Plotly** to identify patterns and trends to help customers make informed investment decisions.

### Table of Contents:
1. **Exercise 1**: Use `yfinance` to Extract Tesla Stock Data
2. **Exercise 2**: Use Webscraping to Extract Tesla Revenue Data
3. **Exercise 3**: Use `yfinance` to Extract GameStop Stock Data
4. **Exercise 4**: Use Webscraping to Extract GameStop Revenue Data
5. **Exercise 5**: Tesla Stock and Revenue Dashboard
6. **Exercise 6**: GameStop Stock and Revenue Dashboard
7. **Bonus Analysis**: Amazon & AMD Stock Data Analysis & Comparative Visualization


In [1]:
# Import required libraries
import yfinance as yf
import pandas as pd
import requests
from bs4 import BeautifulSoup
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import io
import warnings
warnings.filterwarnings("ignore")

print("All libraries imported successfully!")


All libraries imported successfully!


## Define the Graphing Function (`make_graph`)
The function `make_graph` takes two data frames: one for historical stock data (containing `Date` and `Close`) and one for quarterly revenue data (containing `Date` and `Revenue`), along with the name of the stock. It builds an interactive dual-panel dashboard using Plotly subplots.

In [2]:
def make_graph(stock_data, revenue_data, stock):
    """
    Builds an interactive Plotly dashboard showing:
    1. Historical Share Price (Top Subplot)
    2. Historical Quarterly Revenue (Bottom Subplot)
    """
    fig = make_subplots(
        rows=2, cols=1, 
        shared_xaxes=True, 
        subplot_titles=("Historical Share Price", "Historical Revenue"), 
        vertical_spacing=0.25
    )
    
    # Standardize Date formats
    stock_df = stock_data.copy()
    stock_df['Date'] = pd.to_datetime(stock_df['Date'])
    
    revenue_df = revenue_data.copy()
    revenue_df['Date'] = pd.to_datetime(revenue_df['Date'])
    
    # Filter dates up to June 2021 for historical alignment
    stock_data_specific = stock_df[stock_df.Date <= '2021-06-30']
    revenue_data_specific = revenue_df[revenue_df.Date <= '2021-06-30']
    
    # Add Share Price Trace
    fig.add_trace(
        go.Scatter(
            x=stock_data_specific.Date, 
            y=stock_data_specific.Close.astype("float"), 
            name="Share Price ($)",
            line=dict(color="#1f77b4", width=2)
        ), 
        row=1, col=1
    )
    
    # Add Revenue Trace
    fig.add_trace(
        go.Scatter(
            x=revenue_data_specific.Date, 
            y=revenue_data_specific.Revenue.astype("float"), 
            name="Revenue ($M)",
            line=dict(color="#ff7f0e", width=2)
        ), 
        row=2, col=1
    )
    
    # Update layout and axis labels
    fig.update_xaxes(title_text="Date", row=1, col=1)
    fig.update_xaxes(title_text="Date", row=2, col=1)
    fig.update_yaxes(title_text="Price ($US)", row=1, col=1)
    fig.update_yaxes(title_text="Revenue ($US Millions)", row=2, col=1)
    
    fig.update_layout(
        showlegend=True,
        title_text=f"{stock} Historical Share Price vs Quarterly Revenue",
        xaxis_rangeslider_visible=True,
        template="plotly_white",
        height=750,
        width=1000
    )
    
    fig.show()


---
## Exercise 1: Use `yfinance` to Extract Tesla Stock Data

1. Create a `Ticker` object for Tesla using ticker symbol `TSLA`.
2. Extract historical stock information with `period='max'`.
3. Reset the index using `reset_index(inplace=True)`.
4. Display the first five rows using `head()`.


In [3]:
# 1. Create a Ticker object for Tesla
tesla = yf.Ticker("TSLA")

# 2. Extract historical stock data with period set to 'max'
tesla_data = tesla.history(period="max")

# 3. Reset the index of the dataframe in-place
tesla_data.reset_index(inplace=True)

# 4. Display the first five rows of tesla_data
tesla_data.head()


,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits
0,2010-06-29 00:00:00-04:00,1.266667,1.666667,1.169333,1.592667,281494500,0.0,0.0
1,2010-06-30 00:00:00-04:00,1.719333,2.028000,1.553333,1.588667,257806500,0.0,0.0
2,2010-07-01 00:00:00-04:00,1.666667,1.728000,1.351333,1.464000,123282000,0.0,0.0
3,2010-07-02 00:00:00-04:00,1.533333,1.540000,1.247333,1.280000,77097000,0.0,0.0
4,2010-07-06 00:00:00-04:00,1.333333,1.333333,1.055333,1.074000,103003500,0.0,0.0


---
## Exercise 2: Use Webscraping to Extract Tesla Revenue Data

1. Use `requests.get()` to download the Tesla revenue web page from the IBM Skills Network repository.
2. Parse the HTML using `BeautifulSoup`.
3. Extract the `Tesla Quarterly Revenue` table into a Pandas DataFrame.
4. Clean the `Revenue` column by removing commas and dollar signs (`$`).
5. Remove any null or empty string values.
6. Display the last five rows using `tail()`.


In [4]:
# 1. Download the webpage containing Tesla revenue data
url_tesla = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-PY0220EN-SkillsNetwork/labs/project/revenue.htm"
html_data = requests.get(url_tesla).text

# 2. Parse the HTML data using BeautifulSoup
soup = BeautifulSoup(html_data, "html.parser")

# 3. Read HTML tables into a DataFrame (Table 1 corresponds to Tesla Quarterly Revenue)
tesla_revenue = pd.read_html(io.StringIO(html_data))[1]
tesla_revenue.columns = ["Date", "Revenue"]

# 4. Remove comma and dollar sign from Revenue column
tesla_revenue["Revenue"] = tesla_revenue["Revenue"].str.replace(r"[\$,]", "", regex=True)

# 5. Remove null or empty strings from Revenue
tesla_revenue.dropna(inplace=True)
tesla_revenue = tesla_revenue[tesla_revenue["Revenue"] != ""]

# 6. Display the last five rows of tesla_revenue
tesla_revenue.tail()


,Date,Revenue
48,2010-09-30,31
49,2010-06-30,28
50,2010-03-31,21
52,2009-09-30,46
53,2009-06-30,27


---
## Exercise 3: Use `yfinance` to Extract GameStop Stock Data

1. Create a `Ticker` object for GameStop using ticker symbol `GME`.
2. Extract historical stock information with `period='max'`.
3. Reset the index using `reset_index(inplace=True)`.
4. Display the first five rows using `head()`.


In [5]:
# 1. Create a Ticker object for GameStop
gme = yf.Ticker("GME")

# 2. Extract historical stock data with period set to 'max'
gme_data = gme.history(period="max")

# 3. Reset the index of the dataframe in-place
gme_data.reset_index(inplace=True)

# 4. Display the first five rows of gme_data
gme_data.head()


,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits
0,2002-02-13 00:00:00-05:00,1.620128,1.693350,1.603296,1.691666,76216000,0.0,0.0
1,2002-02-14 00:00:00-05:00,1.712707,1.716074,1.670626,1.683250,11021600,0.0,0.0
2,2002-02-15 00:00:00-05:00,1.683250,1.687458,1.658001,1.674834,8389600,0.0,0.0
3,2002-02-19 00:00:00-05:00,1.666418,1.666418,1.578047,1.607504,7410400,0.0,0.0
4,2002-02-20 00:00:00-05:00,1.615920,1.662210,1.603296,1.662210,6892800,0.0,0.0


---
## Exercise 4: Use Webscraping to Extract GameStop Revenue Data

1. Use `requests.get()` to download the GameStop revenue web page.
2. Parse the HTML using `BeautifulSoup`.
3. Extract the `GameStop Quarterly Revenue` table into a Pandas DataFrame.
4. Clean the `Revenue` column by removing commas and dollar signs (`$`).
5. Remove any null or empty string values.
6. Display the last five rows using `tail()`.


In [6]:
# 1. Download the webpage containing GameStop revenue data
url_gme = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-PY0220EN-SkillsNetwork/labs/project/stock.html"
html_data_gme = requests.get(url_gme).text

# 2. Parse the HTML data using BeautifulSoup
soup_gme = BeautifulSoup(html_data_gme, "html.parser")

# 3. Read HTML tables into a DataFrame (Table 1 corresponds to GameStop Quarterly Revenue)
gme_revenue = pd.read_html(io.StringIO(html_data_gme))[1]
gme_revenue.columns = ["Date", "Revenue"]

# 4. Remove comma and dollar sign from Revenue column
gme_revenue["Revenue"] = gme_revenue["Revenue"].str.replace(r"[\$,]", "", regex=True)

# 5. Remove null or empty strings from Revenue
gme_revenue.dropna(inplace=True)
gme_revenue = gme_revenue[gme_revenue["Revenue"] != ""]

# 6. Display the last five rows of gme_revenue
gme_revenue.tail()


,Date,Revenue
57,2006-01-31,1667
58,2005-10-31,534
59,2005-07-31,416
60,2005-04-30,475
61,2005-01-31,709


---
## Exercise 5: Plot Tesla Stock Graph & Dashboard

Call the `make_graph` function with:
- `tesla_data`
- `tesla_revenue`
- `'Tesla'`


In [7]:
# Plot Tesla Stock & Revenue Dashboard
make_graph(tesla_data, tesla_revenue, 'Tesla')


---
## Exercise 6: Plot GameStop Stock Graph & Dashboard

Call the `make_graph` function with:
- `gme_data`
- `gme_revenue`
- `'GameStop'`


In [8]:
# Plot GameStop Stock & Revenue Dashboard
make_graph(gme_data, gme_revenue, 'GameStop')


---
## Extended Analysis: Amazon & AMD Financial Data Analysis

To complete the investment firm's research portfolio across all requested tickers (**Tesla, GameStop, Amazon, AMD**), we analyze Amazon and AMD historical stock performance and comparative trends.


In [9]:
# Extract Amazon and AMD Stock Data
amazon = yf.Ticker("AMZN")
amazon_data = amazon.history(period="max")
amazon_data.reset_index(inplace=True)

amd = yf.Ticker("AMD")
amd_data = amd.history(period="max")
amd_data.reset_index(inplace=True)

print(f"Amazon Historical records: {len(amazon_data)} rows")
print(f"AMD Historical records: {len(amd_data)} rows")

display(amazon_data.head(3))
display(amd_data.head(3))


Amazon Historical records: 7421 rows
AMD Historical records: 11761 rows


,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits
0,1997-05-15 00:00:00-04:00,0.121875,0.125000,0.096354,0.097917,1443120000,0.0,0.0
1,1997-05-16 00:00:00-04:00,0.098438,0.098958,0.085417,0.086458,294000000,0.0,0.0
2,1997-05-19 00:00:00-04:00,0.088021,0.088542,0.081250,0.085417,122136000,0.0,0.0


,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits
0,1980-03-17 00:00:00-05:00,3.12500,3.302083,3.125000,3.145833,219600,0.0,0.0
1,1980-03-18 00:00:00-05:00,3.12500,3.125000,2.937500,3.031250,727200,0.0,0.0
2,1980-03-19 00:00:00-05:00,3.03125,3.083333,3.020833,3.041667,295200,0.0,0.0


### Web Scraping Amazon Historical Financial Data from Lab Source

In [10]:
# Webscrape Amazon historical stock table from IBM Skills Network lab archive
url_amzn = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-PY0220EN-SkillsNetwork/labs/project/amazon_data_webpage.html"
html_amzn = requests.get(url_amzn).text
amzn_tables = pd.read_html(io.StringIO(html_amzn))
amazon_scraped_data = amzn_tables[0]

print("Amazon Scraped Monthly Historical Table:")
amazon_scraped_data.head()


Amazon Scraped Monthly Historical Table:


,Date,Open,High,Low,Close*,Adj Close**,Volume
0,"Jan 01, 2021",3270.00,3363.89,3086.00,3206.20,3206.20,71528900
1,"Dec 01, 2020",3188.50,3350.65,3072.82,3256.93,3256.93,77556200
2,"Nov 01, 2020",3061.74,3366.80,2950.12,3168.04,3168.04,90810500
3,"Oct 01, 2020",3208.00,3496.24,3019.00,3036.15,3036.15,116226100
4,"Sep 01, 2020",3489.58,3552.25,2871.00,3148.73,3148.73,115899300


### Comparative Investment Dashboard: Tesla, GameStop, Amazon & AMD Normalized Performance

In [11]:
# Build an Executive Multi-Stock Normalized Comparison Chart
fig_comp = go.Figure()

# Filter each stock from 2015 onwards to compare on a common baseline
start_date = '2015-01-01'

for name, df, color in [
    ('Tesla (TSLA)', tesla_data, '#e74c3c'),
    ('GameStop (GME)', gme_data, '#2ecc71'),
    ('Amazon (AMZN)', amazon_data, '#f39c12'),
    ('AMD (AMD)', amd_data, '#3498db')
]:
    df_filtered = df[pd.to_datetime(df['Date']) >= start_date].copy()
    initial_price = df_filtered['Close'].iloc[0]
    df_filtered['Normalized_Return'] = (df_filtered['Close'] / initial_price - 1) * 100
    
    fig_comp.add_trace(go.Scatter(
        x=df_filtered['Date'],
        y=df_filtered['Normalized_Return'],
        mode='lines',
        name=name,
        line=dict(color=color, width=2)
    ))

fig_comp.update_layout(
    title='Portfolio Growth Comparison (% Return Since 2015): TSLA vs GME vs AMZN vs AMD',
    xaxis_title='Date',
    yaxis_title='Cumulative Return (%)',
    template='plotly_white',
    height=600,
    width=1000,
    hovermode='x unified'
)

fig_comp.show()


## Summary of Insights for Startup Investment Firm

1. **Tesla (TSLA)**:
   - Experienced explosive share price appreciation particularly beginning in late 2019 / 2020.
   - Revenue trajectory exhibits steep compounding growth, reflecting vehicle production scaling and expanding gross margins.

2. **GameStop (GME)**:
   - High revenue historically generated during Q4 holiday gaming seasons.
   - Characterized by traditional retail revenue decline followed by unprecedented short-squeeze volatility in early 2021 where price decoupled from revenue fundamentals.

3. **Amazon (AMZN) & AMD**:
   - **Amazon** provides stable, sustained mega-cap growth anchored by e-commerce and high-margin AWS cloud infrastructure.
   - **AMD** demonstrates significant turnaround growth from sub-$5 levels prior to 2016 to prominent semiconductor leadership driven by Ryzen and EPYC architecture.
